# MERENVAY — ARC Prize 2026 ARC-AGI-2 offline submission\n\nCompetition wrapper for the answer-isolated candidate. Attach the published MERENVAY offline bundle `.tar.gz` and its `.sha256` sidecar as Kaggle input. This notebook verifies the archive, safely extracts it, checks the internal manifest, runs the answer-isolated builder directly, writes `/kaggle/working/submission.json`, and performs no grading. No ARC Prize Verified claim is implied until ARC Prize/Kaggle returns an official evaluation result.\n

In [ ]:
from pathlib import Path\nimport hashlib, json, os, shutil, subprocess, sys, tarfile\n\nINPUT = Path('/kaggle/input')\nWORK = Path('/kaggle/working')\n\n# 1. Locate and verify the published offline bundle.\nbundles = sorted(INPUT.glob('**/merenvay-arc2-offline-bundle.tar.gz'))\nchecks = sorted(INPUT.glob('**/merenvay-arc2-offline-bundle.tar.gz.sha256'))\nif len(bundles) != 1 or len(checks) != 1:\n    raise RuntimeError(f'Need exactly one bundle and one checksum file; bundles={bundles}, checks={checks}')\nbundle = bundles[0]\nexpected = checks[0].read_text(encoding='utf-8').strip().split()[0].lower()\nh = hashlib.sha256()\nwith bundle.open('rb') as fh:\n    for block in iter(lambda: fh.read(1024*1024), b''):\n        h.update(block)\nactual = h.hexdigest()\nif actual != expected:\n    raise RuntimeError(f'Bundle SHA-256 mismatch: expected {expected}, got {actual}')\nprint('Bundle SHA-256 verified:', actual)\n\n# 2. Safely extract into working storage.\nextract_root = WORK / 'merenvay_bundle_unpack'\nif extract_root.exists():\n    shutil.rmtree(extract_root)\nextract_root.mkdir(parents=True)\nwith tarfile.open(bundle, 'r:gz') as tf:\n    tf.extractall(extract_root, filter='data')\npackage = extract_root / 'package'\nif not (package / 'OFFLINE_BUNDLE_README.md').is_file():\n    raise RuntimeError('Offline package marker missing after extraction')\n\n# 3. Verify every file recorded in the package manifest.\nsubprocess.run(['sha256sum', '-c', 'SHA256SUMS'], cwd=package, check=True)\n\n# 4. Locate the competition challenge JSON, preferring explicit test naming.\npatterns = ['**/arc-agi_test_challenges.json', '**/arc-agi-2_test_challenges.json', '**/*test*challenges*.json', '**/*evaluation*challenges*.json']\ncandidates = []\nfor pattern in patterns:\n    candidates = sorted({p.resolve() for p in INPUT.glob(pattern) if p.is_file()})\n    if candidates:\n        break\nif len(candidates) != 1:\n    raise RuntimeError(f'Expected one ARC competition challenge file, found {len(candidates)}: {candidates}')\nchallenges = candidates[0]\nprint('Challenge source:', challenges)\n\n# 5. Run the answer-isolated builder directly from the verified offline package.\nswipl = package / 'runtime' / 'swi-portable' / 'swipl-portable'\nbuilder = package / 'app' / 'build_submission.py'\nbenchmark = package / 'app' / 'Mentova' / 'src' / 'mentova' / 'games' / 'arc_benchmark_2.pl'\nfor required in (swipl, builder, benchmark):\n    if not required.exists():\n        raise FileNotFoundError(required)\nout = WORK / 'submission.json'\nenv = os.environ.copy()\nenv['PYTHONHASHSEED'] = '0'\nsubprocess.run([sys.executable, str(builder), '--challenges', str(challenges), '--benchmark-file', str(benchmark), '--output', str(out), '--swipl', str(swipl), '--timeout', '39600'], check=True, env=env)\n\n# 6. Validate the competition artifact structurally.\nif not out.exists() or out.stat().st_size == 0:\n    raise RuntimeError('submission.json was not produced')\ndata = json.loads(out.read_text(encoding='utf-8'))\nassert data and all(isinstance(v, list) and len(v) > 0 for v in data.values())\nassert all(all(isinstance(a, dict) and 'attempt_1' in a and 'attempt_2' in a for a in v) for v in data.values())\ndigest = hashlib.sha256(out.read_bytes()).hexdigest()\nprint('tasks:', len(data))\nprint('test_inputs:', sum(len(v) for v in data.values()))\nprint('submission_sha256:', digest)\nprint('output:', out)\nprint('CLAIM_BOUNDARY: official score only after Kaggle/ARC Prize evaluation.')\n